# 🧪 Test Pipeline — SmartSupport RAG

Ce notebook teste chaque module du pipeline RAG étape par étape.
Il sert à valider que chaque composant fonctionne correctement
avant de passer au suivant.

**Ordre des tests :**
1. Loader — chargement des fichiers
2. Cleaner — nettoyage du texte
3. Chunker — découpage en morceaux
4. Embedder — vectorisation
5. Store — indexation ChromaDB
6. RAG — retrieval + génération

In [27]:
# ============================================================
# CELLULE 1 — SETUP
# ============================================================
# On ajoute le dossier racine du projet au PYTHONPATH
# pour que Python trouve nos modules (src/ingestion/loader.py...)
# Sans ça : ModuleNotFoundError
# ============================================================

import sys
sys.path.append('/workspaces/smartsupport-rag')

# Vérification que le setup fonctionne
print('✅ Setup OK — modules accessibles')

✅ Setup OK — modules accessibles


## 📄 PARTIE 1 — Test du Loader

Le loader charge n'importe quel fichier et le convertit en `List[Document]` LangChain.
On teste chaque format supporté : CSV, TXT, JSON, HTML, Email.

In [28]:
# ============================================================
# CELLULE 2 — TEST LOADER CSV
# ============================================================
# On importe la fonction load_file depuis notre module loader
# load_file() détecte l'extension .csv et utilise CSVLoader
# Résultat attendu : 1 Document par ligne du CSV
# Notre fichier a 20 tickets → on attend 20 Documents
# ============================================================

from src.ingestion.loader import load_file

# Chargement du fichier CSV des tickets support
docs_csv = load_file('../data/csv_files/historique_tickets_2026.csv')

# Nombre de documents chargés
# 1 ligne CSV = 1 Document LangChain
print(f'Nombre de documents : {len(docs_csv)}')

# Contenu du premier document
# page_content = le texte extrait de la ligne CSV
print(f'\n--- Contenu du 1er document (300 premiers caractères) ---')
print(docs_csv[0].page_content[:300])

# Métadonnées ajoutées par notre add_metadata()
# filename, file_path, file_type, doc_index
print(f'\n--- Métadonnées ---')
print(docs_csv[0].metadata)

[Loader] ✅ 20 document(s) chargé(s) depuis 'historique_tickets_2026.csv'
Nombre de documents : 20

--- Contenu du 1er document (300 premiers caractères) ---
ticket_id: TK-2026-0001
date_creation: 2026-01-03 08:12
date_resolution: 2026-01-03 09:45
statut: Résolu
priorite: P3
categorie: Compte
sous_categorie: Mot de passe
sujet: Impossible de se connecter après changement de mot de passe
description: Le client indique qu'il a changé son mot de passe hier 

--- Métadonnées ---
{'source': '../data/csv_files/historique_tickets_2026.csv', 'row': 0, 'filename': 'historique_tickets_2026.csv', 'file_path': '../data/csv_files/historique_tickets_2026.csv', 'file_type': '.csv', 'doc_index': 0}


In [29]:
# ============================================================
# CELLULE 3 — TEST LOADER TXT (simule un PDF)
# ============================================================
# Notre FAQ est en .txt — le loader utilise TextLoader
# TextLoader lit tout le fichier comme un seul Document
# Résultat attendu : 1 Document avec tout le contenu de la FAQ
# ============================================================

docs_txt = load_file('../data/pdf_files/FAQ_Support_Client_v3.2.txt')

# TextLoader retourne 1 seul Document (tout le fichier)
print(f'Nombre de documents : {len(docs_txt)}')

# Les 300 premiers caractères du contenu
print(f'\n--- Début du contenu ---')
print(docs_txt[0].page_content[:300])

# Taille totale du document en caractères
# Utile pour estimer combien de chunks on aura après le découpage
print(f'\n--- Taille totale ---')
print(f'{len(docs_txt[0].page_content)} caractères')

print(f'\n--- Métadonnées ---')
print(docs_txt[0].metadata)

[Loader] ✅ 1 document(s) chargé(s) depuis 'FAQ_Support_Client_v3.2.txt'
Nombre de documents : 1

--- Début du contenu ---
FAQ SUPPORT CLIENT — TechVision SAS
Version 3.2 — Mise à jour : Mars 2026
Direction Support & Expérience Client

═══════════════════════════════════════════════════════════════
SECTION 1 — GESTION DES COMPTES ET ACCÈS
═══════════════════════════════════════════════════════════════

Q1. Comment réini

--- Taille totale ---
12165 caractères

--- Métadonnées ---
{'source': '../data/pdf_files/FAQ_Support_Client_v3.2.txt', 'filename': 'FAQ_Support_Client_v3.2.txt', 'file_path': '../data/pdf_files/FAQ_Support_Client_v3.2.txt', 'file_type': '.txt', 'doc_index': 0}


In [30]:
# ============================================================
# CELLULE 4 — TEST LOADER JSON
# ============================================================
# Notre base de connaissances est en JSON
# JSONLoader extrait le contenu avec jq_schema='.'
# qui signifie : extraire tout le contenu JSON
# Résultat attendu : plusieurs Documents selon la structure JSON
# ============================================================

docs_json = load_file('../data/json_files/base_connaissances_techvision.json')

print(f'Nombre de documents : {len(docs_json)}')

# Contenu du premier document JSON extrait
print(f'\n--- Contenu du 1er document ---')
print(docs_json[0].page_content[:400])

print(f'\n--- Métadonnées ---')
print(docs_json[0].metadata)

[Loader] ✅ 4 document(s) chargé(s) depuis 'base_connaissances_techvision.json'
Nombre de documents : 4

--- Contenu du 1er document ---
entreprise:
"TechVision SAS"

--- Métadonnées ---
{'source': '../data/json_files/base_connaissances_techvision.json', 'key': 'entreprise', 'filename': 'base_connaissances_techvision.json', 'file_path': '../data/json_files/base_connaissances_techvision.json', 'file_type': '.json', 'doc_index': 0}


In [31]:
# ============================================================
# CELLULE 5 — TEST LOADER HTML
# ============================================================
# Notre fichier de procédures internes est en HTML
# BSHTMLLoader (BeautifulSoup) retire toutes les balises HTML
# et ne garde que le texte lisible
# Ex: <h2>Procédure</h2><p>Étape 1...</p> → 'Procédure\nÉtape 1...'
# ============================================================

docs_html = load_file('../data/html_files/procedures_support_internes.html')

print(f'Nombre de documents : {len(docs_html)}')

# Vérification que les balises HTML ont bien été supprimées
# On ne doit plus voir de <h1>, <p>, <table>...
print(f'\n--- Contenu (balises HTML supprimées) ---')
print(docs_html[0].page_content[:400])

print(f'\n--- Métadonnées ---')
print(docs_html[0].metadata)

[Loader] ✅ 1 document(s) chargé(s) depuis 'procedures_support_internes.html'
Nombre de documents : 1

--- Contenu (balises HTML supprimées) ---



Procédures Support Internes — TechVision


Procédures Internes Support Client — TechVision SAS
Version : 2.4 | Mise à jour : Février 2026 | Auteur : Direction Support

1. Procédure d'ouverture de ticket
1.1 Canaux de contact et priorités automatiques


Canal
Priorité par défaut
Délai de prise en charge


Chat en direct
P3
2 minutes


Email support@techvision.fr
P3
4 heures


Téléphone (Business

--- Métadonnées ---
{'source': '../data/html_files/procedures_support_internes.html', 'title': 'Procédures Support Internes — TechVision', 'filename': 'procedures_support_internes.html', 'file_path': '../data/html_files/procedures_support_internes.html', 'file_type': '.html', 'doc_index': 0}


In [32]:
# ============================================================
# CELLULE 6 — TEST LOADER EMAIL
# ============================================================
# Nos échanges support sont en .txt (simulent des emails)
# TextLoader lit tout le contenu
# Dans un vrai projet : UnstructuredEmailLoader pour les .eml
# ============================================================

docs_email = load_file('../data/email_files/echanges_support_janvier_2026.txt')

print(f'Nombre de documents : {len(docs_email)}')

# Les 400 premiers caractères — on doit voir les échanges email
print(f'\n--- Contenu ---')
print(docs_email[0].page_content[:400])

print(f'\n--- Métadonnées ---')
print(docs_email[0].metadata)

[Loader] ✅ 1 document(s) chargé(s) depuis 'echanges_support_janvier_2026.txt'
Nombre de documents : 1

--- Contenu ---
From: jean.martin@acme-corp.fr
To: support@techvision.fr
Date: Lundi 6 janvier 2026 09:15
Subject: Problème connexion urgent — équipe commerciale bloquée

Bonjour,

Je suis le responsable IT de la société ACME Corp (client depuis 2 ans, plan Business).
Depuis ce matin 8h30, 12 utilisateurs de notre équipe commerciale ne peuvent plus se connecter à TechVision. Ils reçoivent l'erreur : "Votre sessio

--- Métadonnées ---
{'source': '../data/email_files/echanges_support_janvier_2026.txt', 'filename': 'echanges_support_janvier_2026.txt', 'file_path': '../data/email_files/echanges_support_janvier_2026.txt', 'file_type': '.txt', 'doc_index': 0}


In [33]:
# ============================================================
# CELLULE 7 — TEST LOADER MANUEL (DOCX simulé en TXT)
# ============================================================
# Notre manuel utilisateur est en .txt
# Dans un vrai projet ce serait un .docx chargé par Docx2txtLoader
# ============================================================

docs_manual = load_file('../data/word_files/Manuel_Utilisateur_TechVision_v4.2.txt')

print(f'Nombre de documents : {len(docs_manual)}')
print(f'Taille totale : {len(docs_manual[0].page_content)} caractères')

print(f'\n--- Début du manuel ---')
print(docs_manual[0].page_content[:400])

print(f'\n--- Métadonnées ---')
print(docs_manual[0].metadata)

[Loader] ✅ 1 document(s) chargé(s) depuis 'Manuel_Utilisateur_TechVision_v4.2.txt'
Nombre de documents : 1
Taille totale : 12120 caractères

--- Début du manuel ---
MANUEL UTILISATEUR — TECHVISION PLATEFORME
Version 4.2 — Mars 2026
Document destiné aux utilisateurs finaux et administrateurs

═══════════════════════════════════════════════════════════════
CHAPITRE 1 — PRISE EN MAIN
═══════════════════════════════════════════════════════════════

1.1 Première connexion
──────────────────────
Après réception de votre email d'invitation, suivez ces étapes :

1. C

--- Métadonnées ---
{'source': '../data/word_files/Manuel_Utilisateur_TechVision_v4.2.txt', 'filename': 'Manuel_Utilisateur_TechVision_v4.2.txt', 'file_path': '../data/word_files/Manuel_Utilisateur_TechVision_v4.2.txt', 'file_type': '.txt', 'doc_index': 0}


In [34]:
# ============================================================
# CELLULE 8 — TEST load_directory
# ============================================================
# load_directory() charge TOUS les fichiers d'un dossier
# Elle parcourt le dossier, détecte les formats supportés
# et charge chaque fichier automatiquement
# Utile pour indexer tous les fichiers d'un coup
# ============================================================

from src.ingestion.loader import load_directory

# Charger tous les fichiers CSV du dossier
all_csv = load_directory('../data/csv_files/')
print(f'CSV — Total documents : {len(all_csv)}')

# Charger tous les fichiers PDF (TXT) du dossier
all_pdf = load_directory('../data/pdf_files/')
print(f'PDF — Total documents : {len(all_pdf)}')

# Charger tous les fichiers HTML
all_html = load_directory('../data/html_files/')
print(f'HTML — Total documents : {len(all_html)}')

[Loader] ✅ 20 document(s) chargé(s) depuis 'historique_tickets_2026.csv'
[Loader] 📁 Total : 20 document(s) depuis 'csv_files'
CSV — Total documents : 20
[Loader] ✅ 1 document(s) chargé(s) depuis 'FAQ_Support_Client_v3.2.txt'
[Loader] 📁 Total : 1 document(s) depuis 'pdf_files'
PDF — Total documents : 1
[Loader] ✅ 1 document(s) chargé(s) depuis 'procedures_support_internes.html'
[Loader] 📁 Total : 1 document(s) depuis 'html_files'
HTML — Total documents : 1


In [35]:
# ============================================================
# CELLULE 9 — RÉSUMÉ DES TESTS LOADER
# ============================================================
# On charge tous les fichiers de tous les dossiers
# et on affiche un résumé complet
# C'est ce qu'on fera en production : indexer toutes les sources
# ============================================================

import os

# Liste de tous les dossiers de données
data_folders = [
    '../data/pdf_files/',
    '../data/word_files/',
    '../data/csv_files/',
    '../data/json_files/',
    '../data/html_files/',
    '../data/email_files/',
]

total = 0
print('=== RÉSUMÉ CHARGEMENT PAR SOURCE ===')
print(f'{"Source":<25} {"Documents"}')
print('-' * 35)

for folder in data_folders:
    if os.path.exists(folder):
        # Charger tous les fichiers du dossier
        docs = load_directory(folder)
        folder_name = folder.split('/')[-2]  # nom du dossier
        print(f'{folder_name:<25} {len(docs)}')
        total += len(docs)

print('-' * 35)
print(f'{"TOTAL":<25} {total}')
print(f'\n✅ Loader fonctionne sur tous les formats !')

=== RÉSUMÉ CHARGEMENT PAR SOURCE ===
Source                    Documents
-----------------------------------
[Loader] ✅ 1 document(s) chargé(s) depuis 'FAQ_Support_Client_v3.2.txt'
[Loader] 📁 Total : 1 document(s) depuis 'pdf_files'
pdf_files                 1
[Loader] ✅ 1 document(s) chargé(s) depuis 'Manuel_Utilisateur_TechVision_v4.2.txt'
[Loader] 📁 Total : 1 document(s) depuis 'word_files'
word_files                1
[Loader] ✅ 20 document(s) chargé(s) depuis 'historique_tickets_2026.csv'
[Loader] 📁 Total : 20 document(s) depuis 'csv_files'
csv_files                 20
[Loader] ✅ 4 document(s) chargé(s) depuis 'base_connaissances_techvision.json'
[Loader] 📁 Total : 4 document(s) depuis 'json_files'
json_files                4
[Loader] ✅ 1 document(s) chargé(s) depuis 'procedures_support_internes.html'
[Loader] 📁 Total : 1 document(s) depuis 'html_files'
html_files                1
[Loader] ✅ 1 document(s) chargé(s) depuis 'echanges_support_janvier_2026.txt'
[Loader] 📁 Total : 1 doc

In [36]:
# ============================================================
# CELLULE ANALYSE — Tailles des documents avant chunking
# ============================================================
# On analyse la taille de chaque document chargé
# pour vérifier si les paramètres de chunking sont adaptés
# ============================================================

from src.ingestion.loader import load_directory
import os

data_folders = [
    '../data/pdf_files/',
    '../data/word_files/',
    '../data/csv_files/',
    '../data/json_files/',
    '../data/html_files/',
    '../data/email_files/',
]

print("=== ANALYSE DES TAILLES DE DOCUMENTS ===\n")
print(f"{'Source':<15} {'Doc':<40} {'Caractères':>12} {'Tokens (~)':>12} {'Chunks (~)':>12}")
print("-" * 95)

total_docs = 0
total_chunks_estimate = 0

for folder in data_folders:
    if not os.path.exists(folder):
        continue

    docs = load_directory(folder)
    folder_name = folder.split('/')[-2]

    for doc in docs:
        chars = len(doc.page_content)
        # Estimation : 1 token ≈ 4 caractères en français
        tokens = chars // 4
        # Estimation chunks avec chunk_size=600, overlap=100
        chunk_size = 600
        overlap = 100
        chunks_estimate = max(1, (tokens - overlap) // (chunk_size - overlap))

        filename = doc.metadata.get('filename', 'inconnu')[:38]
        print(f"{folder_name:<15} {filename:<40} {chars:>12,} {tokens:>12,} {chunks_estimate:>12,}")

        total_docs += 1
        total_chunks_estimate += chunks_estimate

print("-" * 95)
print(f"\nTotal documents  : {total_docs}")
print(f"Total chunks (~) : {total_chunks_estimate}")
print(f"\n💡 Ces estimations sont approximatives — le chunker réel sera plus précis")

=== ANALYSE DES TAILLES DE DOCUMENTS ===

Source          Doc                                        Caractères   Tokens (~)   Chunks (~)
-----------------------------------------------------------------------------------------------
[Loader] ✅ 1 document(s) chargé(s) depuis 'FAQ_Support_Client_v3.2.txt'
[Loader] 📁 Total : 1 document(s) depuis 'pdf_files'
pdf_files       FAQ_Support_Client_v3.2.txt                    12,165        3,041            5
[Loader] ✅ 1 document(s) chargé(s) depuis 'Manuel_Utilisateur_TechVision_v4.2.txt'
[Loader] 📁 Total : 1 document(s) depuis 'word_files'
word_files      Manuel_Utilisateur_TechVision_v4.2.txt         12,120        3,030            5
[Loader] ✅ 20 document(s) chargé(s) depuis 'historique_tickets_2026.csv'
[Loader] 📁 Total : 20 document(s) depuis 'csv_files'
csv_files       historique_tickets_2026.csv                       675          168            1
csv_files       historique_tickets_2026.csv                       637          159          

In [37]:
# ============================================================
# CELLULE ANALYSE LOADER — Tailles des documents chargés
# Sans estimation de chunks — ce sera fait dans chunker.py
# ============================================================

from src.ingestion.loader import load_file
import os

files = [
    ('pdf',   '../data/pdf_files/FAQ_Support_Client_v3.2.txt'),
    ('word',  '../data/word_files/Manuel_Utilisateur_TechVision_v4.2.txt'),
    ('csv',   '../data/csv_files/historique_tickets_2026.csv'),
    ('json',  '../data/json_files/base_connaissances_techvision.json'),
    ('html',  '../data/html_files/procedures_support_internes.html'),
    ('email', '../data/email_files/echanges_support_janvier_2026.txt'),
]

print("=== RÉSUMÉ LOADER ===\n")
print(f"{'Source':<10} {'Fichier':<45} {'Docs':>6} {'Chars':>10} {'Tokens':>10}")
print("-" * 85)

total_docs  = 0
total_chars = 0

for source, path in files:
    docs = load_file(path)
    for doc in docs:
        chars  = len(doc.page_content)
        tokens = chars // 4
        filename = os.path.basename(path)[:43]
        print(f"{source:<10} {filename:<45} {1:>6} {chars:>10,} {tokens:>10,}")
        total_docs  += 1
        total_chars += chars

print("-" * 85)
print(f"\nTotal documents : {total_docs}")
print(f"Total caractères : {total_chars:,}")
print(f"\n✅ Loader OK — les chunks seront calculés dans chunker.py")

=== RÉSUMÉ LOADER ===

Source     Fichier                                         Docs      Chars     Tokens
-------------------------------------------------------------------------------------
[Loader] ✅ 1 document(s) chargé(s) depuis 'FAQ_Support_Client_v3.2.txt'
pdf        FAQ_Support_Client_v3.2.txt                        1     12,165      3,041
[Loader] ✅ 1 document(s) chargé(s) depuis 'Manuel_Utilisateur_TechVision_v4.2.txt'
word       Manuel_Utilisateur_TechVision_v4.2.txt             1     12,120      3,030
[Loader] ✅ 20 document(s) chargé(s) depuis 'historique_tickets_2026.csv'
csv        historique_tickets_2026.csv                        1        675        168
csv        historique_tickets_2026.csv                        1        637        159
csv        historique_tickets_2026.csv                        1        674        168
csv        historique_tickets_2026.csv                        1        663        165
csv        historique_tickets_2026.csv                       

## 📄 PARTIE 2 — Test du Cleaner

Le Cleaner nettoie n'importe quel fichier et le convertit en `List[Document]` LangChain.
On teste chaque format supporté : CSV, TXT, JSON, HTML, Email.

In [ ]:
# ============================================================
# TEST NETTOYAGE FAQ
# SECTION 2 — TEST CLEANER 
# ============================================================
# On importe clean_documents depuis cleaner.py
# On compare le texte avant et après nettoyage
# pour vérifier que le nettoyage fonctionne correctement
# ============================================================

import importlib
import src.ingestion.cleaner as cleaner_module
importlib.reload(cleaner_module)
from src.ingestion.cleaner import clean_documents
from src.ingestion.loader import load_file

# Chargement de la FAQ
docs_faq = load_file('../data/pdf_files/FAQ_Support_Client_v3.2.txt')

print("=== AVANT NETTOYAGE ===")
# Taille du texte brut sorti du loader
print(f"Taille : {len(docs_faq[0].page_content)} caractères")
# Les 200 premiers caractères bruts
print(docs_faq[0].page_content[:200])

# Nettoyage
cleaned_faq = clean_documents(docs_faq)

print("\n=== APRÈS NETTOYAGE ===")
# Taille après nettoyage — doit être légèrement réduite
print(f"Taille : {len(cleaned_faq[0].page_content)} caractères")
# Les 200 premiers caractères propres
print(cleaned_faq[0].page_content[:200])
# Vérification que cleaned=True est dans les métadonnées
print(f"\nMétadonnées : {cleaned_faq[0].metadata}")

[Loader] ✅ 1 document(s) chargé(s) depuis 'FAQ_Support_Client_v3.2.txt'
=== AVANT NETTOYAGE ===
Taille : 12165 caractères
FAQ SUPPORT CLIENT — TechVision SAS
Version 3.2 — Mise à jour : Mars 2026
Direction Support & Expérience Client

═══════════════════════════════════════════════════════════════
SECTION 1 — GESTION DES
[Cleaner] ✅ 1 document(s) propres (0 supprimé(s) car vides ou trop courts)

=== APRÈS NETTOYAGE ===
Taille : 12137 caractères
FAQ SUPPORT CLIENT — TechVision SAS
Version 3.2 — Mise à jour : Mars 2026
Direction Support & Expérience Client

═══════════════════════════════════════════════════════════════
SECTION 1 — GESTION DES

Métadonnées : {'source': '../data/pdf_files/FAQ_Support_Client_v3.2.txt', 'filename': 'FAQ_Support_Client_v3.2.txt', 'file_path': '../data/pdf_files/FAQ_Support_Client_v3.2.txt', 'file_type': '.txt', 'doc_index': 0, 'cleaned': True}


In [40]:
# ============================================================
# TEST CLEANER — CSV tickets
# ============================================================
# Le CSV a 20 documents — on vérifie qu'aucun n'est supprimé
# car chaque ticket a suffisamment de contenu
# ============================================================

docs_csv = load_file('../data/csv_files/historique_tickets_2026.csv')

print("=== CSV — AVANT / APRÈS ===")
print(f"Avant nettoyage : {len(docs_csv)} documents")

cleaned_csv = clean_documents(docs_csv)

print(f"Après nettoyage : {len(cleaned_csv)} documents")
print(f"\nContenu du 1er ticket nettoyé :")
print(cleaned_csv[0].page_content[:300])

[Loader] ✅ 20 document(s) chargé(s) depuis 'historique_tickets_2026.csv'
=== CSV — AVANT / APRÈS ===
Avant nettoyage : 20 documents
[Cleaner] ✅ 20 document(s) propres (0 supprimé(s) car vides ou trop courts)
Après nettoyage : 20 documents

Contenu du 1er ticket nettoyé :
ticket_id: TK-2026-0001
date_creation: 2026-01-03 08:12
date_resolution: 2026-01-03 09:45
statut: Résolu
priorite: P3
categorie: Compte
sous_categorie: Mot de passe
sujet: Impossible de se connecter après changement de mot de passe
description: Le client indique qu'il a changé son mot de passe hier 


In [41]:
# ============================================================
# TEST CLEANER — Résumé toutes sources
# ============================================================
# On charge et nettoie tous les dossiers
# On affiche combien de documents ont été conservés
# et combien ont été supprimés car vides ou trop courts
# ============================================================

from src.ingestion.loader import load_directory

print("=== RÉSUMÉ CLEANER — TOUTES SOURCES ===\n")
print(f"{'Source':<15} {'Avant':>8} {'Après':>8} {'Supprimés':>10}")
print("-" * 45)

folders = [
    ('pdf',   '../data/pdf_files/'),
    ('word',  '../data/word_files/'),
    ('csv',   '../data/csv_files/'),
    ('json',  '../data/json_files/'),
    ('html',  '../data/html_files/'),
    ('email', '../data/email_files/'),
]

total_avant = 0
total_apres = 0

for source, folder in folders:
    # Chargement
    docs = load_directory(folder)
    # Nettoyage
    cleaned = clean_documents(docs)
    # Calcul des suppressions
    suppr = len(docs) - len(cleaned)

    print(f"{source:<15} {len(docs):>8} {len(cleaned):>8} {suppr:>10}")
    total_avant += len(docs)
    total_apres += len(cleaned)

print("-" * 45)
print(f"{'TOTAL':<15} {total_avant:>8} {total_apres:>8} {total_avant - total_apres:>10}")
print(f"\n✅ Cleaner OK — prêt pour chunker.py")

=== RÉSUMÉ CLEANER — TOUTES SOURCES ===

Source             Avant    Après  Supprimés
---------------------------------------------
[Loader] ✅ 1 document(s) chargé(s) depuis 'FAQ_Support_Client_v3.2.txt'
[Loader] 📁 Total : 1 document(s) depuis 'pdf_files'
[Cleaner] ✅ 1 document(s) propres (0 supprimé(s) car vides ou trop courts)
pdf                    1        1          0
[Loader] ✅ 1 document(s) chargé(s) depuis 'Manuel_Utilisateur_TechVision_v4.2.txt'
[Loader] 📁 Total : 1 document(s) depuis 'word_files'
[Cleaner] ✅ 1 document(s) propres (0 supprimé(s) car vides ou trop courts)
word                   1        1          0
[Loader] ✅ 20 document(s) chargé(s) depuis 'historique_tickets_2026.csv'
[Loader] 📁 Total : 20 document(s) depuis 'csv_files'
[Cleaner] ✅ 20 document(s) propres (0 supprimé(s) car vides ou trop courts)
csv                   20       20          0
[Loader] ✅ 4 document(s) chargé(s) depuis 'base_connaissances_techvision.json'
[Loader] 📁 Total : 4 document(s) depuis 'j

In [42]:
# Cellule diagnostic — quel doc JSON est supprimé ?
from src.ingestion.loader import load_file
from src.ingestion.cleaner import clean_documents

docs_json = load_file('../data/json_files/base_connaissances_techvision.json')

print("=== DIAGNOSTIC JSON ===")
for i, doc in enumerate(docs_json):
    print(f"\nDoc {i} — {len(doc.page_content)} caractères")
    print(f"Clé : {doc.metadata.get('key', '?')}")
    print(f"Contenu : {doc.page_content[:100]}")

[Loader] ✅ 4 document(s) chargé(s) depuis 'base_connaissances_techvision.json'
=== DIAGNOSTIC JSON ===

Doc 0 — 28 caractères
Clé : entreprise
Contenu : entreprise:
"TechVision SAS"

Doc 1 — 14 caractères
Clé : version
Contenu : version:
"2.1"

Doc 2 — 34 caractères
Clé : derniere_mise_a_jour
Contenu : derniere_mise_a_jour:
"2026-03-01"

Doc 3 — 7446 caractères
Clé : base_de_connaissances
Contenu : base_de_connaissances:
{
  "plans_tarifaires": {
    "starter": {
      "prix_mensuel": 29,
      "p
